# Prepared CSV demo (synthetic inputs)

This smaller example uses simulated CT features. For a real CT, use **quickstart.ipynb**.


In [ ]:
from pathlib import Path
import os

# Works when Jupyter starts in the repository or in notebooks/.
root = Path.cwd() if (Path.cwd() / "data/sample.csv").exists() else Path.cwd().parent
if not (root / "data/sample.csv").exists():
    raise FileNotFoundError("Start Jupyter from the repository folder.")
os.chdir(root)
print("Working folder:", root.name)


## 1. Download the model

The downloader verifies checksums and saves the model in `weights/`. Previously downloaded, correct files are reused. No account is needed.


In [ ]:
from nsclc_toxicity.download import download_weights
download_weights(Path("weights"))


## 2. Read the sample CSV

Each patient has a baseline row and optional follow-up rows. Clinical and quantitative CT features repeat the baseline values. The DL hazard columns are simulated inputs in this example.


In [ ]:
import pandas as pd

sample = pd.read_csv("data/sample.csv")
print(f"{sample.patient_id.nunique()} fictional patients; {len(sample)} scans")
sample[["patient_id", "scan_id", "scan_day", "is_baseline"]].head()


## 3. Run inference

`Multimodal` is the final three-model ensemble. Predictions are limited to time points after the scan.


In [ ]:
from nsclc_toxicity.model import load_model, predict

model = load_model("weights/final_model.json")
predictions = predict(sample, model)
Path("results").mkdir(exist_ok=True)
predictions.to_csv("results/predictions.csv", index=False)
final = predictions.loc[predictions["model"].eq("Multimodal")]
final.head(8)


## 4. Compare future time points

Values are model scores, not calibrated patient risk percentages. Changes in fusion weights mean scores do not have to rise with time.


In [ ]:
table = final.pivot(index=["patient_id", "scan_id"], columns="horizon_days", values="score")
table.columns = [f"Score at day {int(day)}" for day in table.columns]
table.head(8).round(3)


## 5. Repeat the experiment on the synthetic sample

This fits the tabular branches within patient folds and evaluates the selected fusion with simulated CT outputs. For your own CT retraining, follow `docs/CT_WORKFLOW.md`.


In [ ]:
from nsclc_toxicity.training import run_experiment

report = run_experiment(sample, Path("results/notebook_experiment"))
pd.read_csv("results/notebook_experiment/performance.csv").round(3)


To analyze your own prepared CSV, replace `data/sample.csv` with its local path. See `data/data_dictionary.csv`. Keep patient data in `private_data/` and generated outputs in `results/`; both folders are ignored by Git.
